# Notebook 1: Transformer Fundamentals

In this notebook you will implement the core building blocks of a Transformer from scratch:

1. **Scaled Dot-Product Attention** (SDPA)
2. **Multi-Head Self-Attention** (MHA)
3. **Feed-Forward MLP with GELU**
4. **Sinusoidal Positional / Timestep Embeddings**
5. **Pre-LN Transformer Block**

Each exercise provides helper code and test cases with `assert` statements. Fill in every `# TODO` block, then run the test cell to verify.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
DEVICE = "cpu"

---
## Exercise 1a: Scaled Dot-Product Attention

Given query $Q$, key $K$, and value $V$ matrices, compute:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

where $d_k$ is the dimension of the key vectors (last dim of $K$).

Optionally apply dropout to the attention weights.

In [ ]:
def scaled_dot_product_attention(
    Q: torch.Tensor,   # (..., seq_len_q, d_k)
    K: torch.Tensor,   # (..., seq_len_k, d_k)
    V: torch.Tensor,   # (..., seq_len_k, d_v)
    dropout_p: float = 0.0,
    training: bool = False,
) -> torch.Tensor:
    """
    Compute scaled dot-product attention.

    Returns
    -------
    Tensor of shape (..., seq_len_q, d_v)
    """
    # TODO: Implement scaled dot-product attention
    # Steps:
    # 1. Compute d_k from the last dimension of K
    # 2. Compute attention scores: Q @ K^T / sqrt(d_k)
    # 3. Apply softmax over the last dimension
    # 4. Optionally apply dropout (only when training=True)
    # 5. Multiply by V and return
    raise NotImplementedError("Implement scaled_dot_product_attention")

In [ ]:
# === Tests for Exercise 1a ===
torch.manual_seed(0)
B, H, T, D = 2, 4, 8, 16
Q = torch.randn(B, H, T, D)
K = torch.randn(B, H, T, D)
V = torch.randn(B, H, T, D)

out = scaled_dot_product_attention(Q, K, V)

# Shape check
assert out.shape == (B, H, T, D), f"Expected shape {(B, H, T, D)}, got {out.shape}"

# Compare against PyTorch reference
ref = F.scaled_dot_product_attention(Q, K, V)
assert torch.allclose(out, ref, atol=1e-5), (
    f"Output differs from F.scaled_dot_product_attention, max diff: {(out - ref).abs().max().item():.6e}"
)

# Verify attention weights sum to 1
d_k = D
scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
weights = scores.softmax(dim=-1)
assert torch.allclose(weights.sum(dim=-1), torch.ones(B, H, T), atol=1e-5), "Attention weights should sum to 1"

print("Exercise 1a passed!")

---
## Exercise 1b: Multi-Head Self-Attention

Implement the full Multi-Head Self-Attention module. Architecture:

1. A single linear layer projects input $x$ into $Q$, $K$, $V$ concatenated: `qkv = W_qkv @ x`, producing a tensor of shape `[B, T, 3*dim]`
2. Reshape to `[3, B, num_heads, T, head_dim]` and unbind into Q, K, V
3. Compute scaled dot-product attention per head
4. Concatenate heads and apply output projection

$$\text{MultiHead}(x) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$
$$\text{where head}_i = \text{Attention}(x W_i^Q, x W_i^K, x W_i^V)$$

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """
    Multi-Head Self-Attention.

    Parameters
    ----------
    dim : int
        Model / embedding dimension.
    num_heads : int
        Number of attention heads. dim must be divisible by num_heads.
    qkv_bias : bool
        Whether to include bias in the QKV projection.
    attn_drop : float
        Dropout rate on attention weights.
    proj_drop : float
        Dropout rate on the output projection.
    """

    def __init__(
        self,
        dim: int,
        num_heads: int = 8,
        qkv_bias: bool = False,
        attn_drop: float = 0.0,
        proj_drop: float = 0.0,
    ):
        super().__init__()
        assert dim % num_heads == 0, "dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        # TODO: Define the layers:
        # self.qkv   -> nn.Linear projecting dim to 3*dim (with bias=qkv_bias)
        # self.attn_drop -> nn.Dropout(attn_drop)
        # self.proj   -> nn.Linear projecting dim to dim
        # self.proj_drop -> nn.Dropout(proj_drop)
        raise NotImplementedError("Define layers in __init__")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : Tensor of shape [B, T, dim]

        Returns
        -------
        Tensor of shape [B, T, dim]
        """
        B, T, C = x.shape

        # TODO: Implement the forward pass:
        # 1. Project: qkv = self.qkv(x)  -> [B, T, 3*C]
        # 2. Reshape to [B, T, 3, num_heads, head_dim]
        # 3. Permute to [3, B, num_heads, T, head_dim]
        # 4. Unbind dim 0 to get q, k, v each [B, num_heads, T, head_dim]
        # 5. Compute attention: scores = (q @ k^T) * scale, softmax, dropout, @ v
        # 6. Transpose back to [B, T, C] and apply self.proj + self.proj_drop
        raise NotImplementedError("Implement forward")

In [ ]:
# === Tests for Exercise 1b ===
torch.manual_seed(1)
B, T, D, H = 2, 10, 64, 8
mha = MultiHeadSelfAttention(dim=D, num_heads=H, qkv_bias=True).eval()
x = torch.randn(B, T, D)
out = mha(x)

# Shape check
assert out.shape == (B, T, D), f"Expected {(B, T, D)}, got {out.shape}"

# Verify it's a proper linear function of input (no stochasticity in eval mode)
out2 = mha(x)
assert torch.allclose(out, out2, atol=1e-6), "Deterministic in eval mode"

# Gradient check
x_g = torch.randn(B, T, D, requires_grad=True)
loss = mha(x_g).sum()
loss.backward()
assert x_g.grad is not None and x_g.grad.shape == (B, T, D), "Gradients must flow"

# Verify that qkv projection has correct sizes
assert mha.qkv.weight.shape == (3 * D, D), f"qkv weight shape wrong: {mha.qkv.weight.shape}"
assert mha.proj.weight.shape == (D, D), f"proj weight shape wrong: {mha.proj.weight.shape}"

print("Exercise 1b passed!")

---
## Exercise 1c: MLP Block with GELU

The standard Transformer feed-forward sub-layer:

$$\text{FFN}(x) = W_2 \, \text{GELU}(W_1 x + b_1) + b_2$$

The hidden dimension is typically `mlp_ratio * dim` (e.g. 4x expansion).

Use `nn.GELU(approximate='tanh')` for the activation.

In [ ]:
class MlpBlock(nn.Module):
    """
    Two-layer MLP with GELU activation.

    Parameters
    ----------
    in_features : int
    hidden_features : int or None
        If None, defaults to in_features.
    out_features : int or None
        If None, defaults to in_features.
    drop : float
        Dropout rate applied after each linear layer.
    """

    def __init__(
        self,
        in_features: int,
        hidden_features: int = None,
        out_features: int = None,
        drop: float = 0.0,
    ):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features

        # TODO: Define:
        # self.fc1  -> nn.Linear(in_features, hidden_features)
        # self.act  -> nn.GELU(approximate='tanh')
        # self.fc2  -> nn.Linear(hidden_features, out_features)
        # self.drop -> nn.Dropout(drop)
        raise NotImplementedError("Define layers")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: x -> fc1 -> act -> drop -> fc2 -> drop -> return
        raise NotImplementedError("Implement forward")

In [ ]:
# === Tests for Exercise 1c ===
torch.manual_seed(2)
D = 64
mlp = MlpBlock(in_features=D, hidden_features=D * 4).eval()
x = torch.randn(2, 10, D)
out = mlp(x)

assert out.shape == x.shape, f"Expected {x.shape}, got {out.shape}"

# Check layer sizes
assert mlp.fc1.weight.shape == (D * 4, D), f"fc1 shape: {mlp.fc1.weight.shape}"
assert mlp.fc2.weight.shape == (D, D * 4), f"fc2 shape: {mlp.fc2.weight.shape}"

# GELU activation should differ from ReLU
x_neg = torch.tensor([-0.5])
gelu_val = mlp.act(x_neg)
assert gelu_val.item() < 0, "GELU(-0.5) should be slightly negative (not zero like ReLU)"

# Deterministic in eval
assert torch.allclose(mlp(x), out, atol=1e-6)

print("Exercise 1c passed!")

---
## Exercise 1d: Sinusoidal Timestep Embedding

Used in both positional encoding (Transformers) and timestep conditioning (diffusion/flow models).

Given a scalar timestep $t$ and embedding dimension $d$:

$$\text{freq}_i = \exp\!\left(-\frac{\ln(\text{max\_period})}{d/2} \cdot i\right), \quad i = 0, \dots, d/2 - 1$$

$$\text{emb}(t) = \big[\cos(t \cdot \text{freq}_0), \dots, \cos(t \cdot \text{freq}_{d/2-1}),\; \sin(t \cdot \text{freq}_0), \dots, \sin(t \cdot \text{freq}_{d/2-1})\big]$$

If $d$ is odd, pad with a trailing zero.

In [ ]:
def timestep_embedding(
    t: torch.Tensor,    # [B] — batch of scalar timesteps
    dim: int,            # embedding dimension
    max_period: int = 100,
) -> torch.Tensor:
    """
    Create sinusoidal timestep embeddings.

    Parameters
    ----------
    t : Tensor of shape [B]
    dim : int
    max_period : int

    Returns
    -------
    Tensor of shape [B, dim]
    """
    # TODO: Implement sinusoidal timestep embedding
    # Steps:
    # 1. half = dim // 2
    # 2. Compute frequency vector: freqs = exp(-ln(max_period) * arange(0, half) / half)
    # 3. Compute args = t[:, None] * freqs[None]  -> [B, half]
    # 4. Concatenate [cos(args), sin(args)] along dim=-1 -> [B, dim] (or [B, 2*half])
    # 5. If dim is odd, pad with a zero column
    raise NotImplementedError("Implement timestep_embedding")

In [ ]:
# === Tests for Exercise 1d ===
t = torch.tensor([0.0, 0.5, 1.0])

# Even dimension
emb = timestep_embedding(t, dim=32, max_period=100)
assert emb.shape == (3, 32), f"Expected (3, 32), got {emb.shape}"

# At t=0, cos(0)=1 for all frequencies, sin(0)=0
emb_zero = timestep_embedding(torch.tensor([0.0]), dim=16, max_period=100)
assert torch.allclose(emb_zero[0, :8], torch.ones(8), atol=1e-5), "cos(0) should be 1"
assert torch.allclose(emb_zero[0, 8:], torch.zeros(8), atol=1e-5), "sin(0) should be 0"

# Odd dimension -- should have trailing zero
emb_odd = timestep_embedding(torch.tensor([1.0]), dim=33, max_period=100)
assert emb_odd.shape == (1, 33), f"Expected (1, 33), got {emb_odd.shape}"
assert emb_odd[0, -1].item() == 0.0, "Last element should be 0 for odd dim"

# Different timesteps should give different embeddings
emb_a = timestep_embedding(torch.tensor([0.1]), dim=32)
emb_b = timestep_embedding(torch.tensor([0.9]), dim=32)
assert not torch.allclose(emb_a, emb_b, atol=1e-3), "Different timesteps must produce different embeddings"

print("Exercise 1d passed!")

---
## Exercise 1e: Pre-LN Transformer Block

The standard pre-LayerNorm Transformer block used in ViT and many modern architectures:

$$x = x + \text{Attn}(\text{LN}_1(x))$$
$$x = x + \text{MLP}(\text{LN}_2(x))$$

Assemble your `MultiHeadSelfAttention` and `MlpBlock` into this block.

In [ ]:
class PreLNTransformerBlock(nn.Module):
    """
    Pre-LayerNorm Transformer block.

    Parameters
    ----------
    dim : int
        Hidden dimension.
    num_heads : int
        Number of attention heads.
    mlp_ratio : float
        MLP hidden dim = dim * mlp_ratio.
    """

    def __init__(self, dim: int, num_heads: int, mlp_ratio: float = 4.0):
        super().__init__()
        # TODO: Define:
        # self.norm1 -> nn.LayerNorm(dim)
        # self.norm2 -> nn.LayerNorm(dim)
        # self.attn  -> MultiHeadSelfAttention(dim, num_heads, qkv_bias=True, attn_drop=0.1)
        # self.mlp   -> MlpBlock(dim, hidden_features=int(dim * mlp_ratio), drop=0.1)
        raise NotImplementedError("Define layers")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : Tensor [B, T, dim]

        Returns
        -------
        Tensor [B, T, dim]
        """
        # TODO: Implement pre-LN forward:
        # x = x + self.attn(self.norm1(x))
        # x = x + self.mlp(self.norm2(x))
        # return x
        raise NotImplementedError("Implement forward")

In [ ]:
# === Tests for Exercise 1e ===
torch.manual_seed(3)
B, T, D, H = 2, 10, 64, 8
block = PreLNTransformerBlock(dim=D, num_heads=H, mlp_ratio=4.0).eval()
x = torch.randn(B, T, D)
out = block(x)

# Shape preservation (residual connection)
assert out.shape == (B, T, D), f"Expected {(B, T, D)}, got {out.shape}"

# Residual: output should differ from input but not be wildly different
diff = (out - x).abs().mean().item()
assert diff > 1e-4, "Block should modify the input"
assert diff < 100.0, "Block output shouldn't explode"

# Deterministic in eval
assert torch.allclose(block(x), out, atol=1e-6), "Should be deterministic in eval mode"

# Gradient flow through residual
x_g = torch.randn(B, T, D, requires_grad=True)
block.train()
loss = block(x_g).sum()
loss.backward()
assert x_g.grad is not None, "Gradients must flow"
assert (x_g.grad.abs() > 0).any(), "Non-zero gradients expected"

# Stacking multiple blocks shouldn't error
stack = nn.Sequential(*[PreLNTransformerBlock(D, H) for _ in range(4)]).eval()
out_stack = stack(torch.randn(1, 5, D))
assert out_stack.shape == (1, 5, D), "Stacked blocks should preserve shape"

print("Exercise 1e passed!")
print("\n=== All Notebook 1 exercises passed! ===")